# 01 — Fetch data

Downloads Austrian PegelAlarm station metadata, complete hourly water-level histories for the selected station network, and matching hourly GeoSphere INCA analysis weather. All timestamps and request boundaries are UTC.

**Inputs:** PegelAlarm API (credentials from `.env`) and GeoSphere Austria INCA historical timeseries API  
**Outputs:** independent Parquet artifacts in `data/raw/`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from src.basic_api_access import BasicApiAccess
from src.config import (
    COUNTRY_CODE,
    GRANULARITY,
    PEGELALARM_PASSWORD,
    PEGELALARM_USERNAME,
    SKIP_IF_EXISTS,
    STATION_IDS,
    UNIT,
)
from src.fetch_data import (
    fetch_hourly_history,
    fetch_inca,
    find_archive_start,
    flatten_station_catalog,
    resolve_station_coordinates,
    summarize_failures,
    utc_current_hour,
)

if not PEGELALARM_USERNAME or not PEGELALARM_PASSWORD:
    raise RuntimeError(
        "Set PEGELALARM_USERNAME and PEGELALARM_PASSWORD in .env before fetching data"
    )

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
api = BasicApiAccess(PEGELALARM_USERNAME, PEGELALARM_PASSWORD)
failures = {}

## Austrian station catalog

The catalog includes every Austrian measurement station. Nested collections are omitted so the raw metadata remains a flat, Parquet-compatible table.

In [ ]:
catalog_path = RAW_DIR / f"pegelalarm_stations_{COUNTRY_CODE.lower()}.parquet"
catalog = None

try:
    if SKIP_IF_EXISTS and catalog_path.exists():
        catalog = pd.read_parquet(catalog_path)
        print(f"Skipping existing station catalog: {catalog_path}")
    else:
        payload = api.query_current_data(country_code=COUNTRY_CODE)
        catalog = flatten_station_catalog(payload)
        catalog.to_parquet(catalog_path, index=False)
        print(f"Saved {len(catalog):,} stations to {catalog_path}")
except Exception as error:
    failures[str(catalog_path)] = error
    print(f"Failed {catalog_path}: {error}")

## Target and upstream station network

The target and upstream stations are configured in `src/config.py` and fetched in the listed order.

In [ ]:
station_ids = list(STATION_IDS)
print(f"Using {len(station_ids)} configured stations: {', '.join(station_ids)}")

## Per-station water-level histories

A broad yearly PegelAlarm request locates each station's archive start. Hourly measurements are then downloaded in non-overlapping eight-day chunks.

In [ ]:
end_utc = utc_current_hour()
water_by_station = {}
failures = {}
latest_water = None

for station_id in tqdm(station_ids, desc="Water histories", unit="station"):
    water_path = RAW_DIR / f"pegelalarm_{station_id}_height_hour.parquet"
    water = None

    try:
        if SKIP_IF_EXISTS and water_path.exists():
            water = pd.read_parquet(water_path)
            print(f"Skipping existing water history: {water_path}")
        else:
            start_utc = find_archive_start(api, station_id, end_utc, unit=UNIT)
            water = fetch_hourly_history(
                api,
                station_id,
                start_utc,
                end_utc,
                unit=UNIT,
                granularity=GRANULARITY,
            )
            water.to_parquet(water_path, index=False)
            print(f"Saved {len(water):,} water observations to {water_path}")
        water_by_station[station_id] = water
        latest_water = water
    except Exception as error:
        failures[str(water_path)] = error
        print(f"Failed {water_path}: {error}")

if latest_water is not None:
    latest_water.info()
    display(latest_water.head())
else:
    print("No water history DataFrame is available to display.")

## Per-station weather histories

GeoSphere INCA is queried once per station over its available water-level range. The historical analysis uses a 1 km grid and hourly UTC timestamps; each query returns the nearest grid point.

In [ ]:
latest_weather = None
failures = {}
for station_id in tqdm(station_ids, desc="Weather histories", unit="station"):
    weather_path = RAW_DIR / f"geosphere_inca_{station_id}_hour.parquet"

    try:
        if SKIP_IF_EXISTS and weather_path.exists():
            weather = pd.read_parquet(weather_path)
            print(f"Skipping existing weather history: {weather_path}")
        else:
            if catalog is None:
                raise RuntimeError("Austrian station metadata is unavailable")
            water = water_by_station.get(station_id)
            if water is None:
                raise RuntimeError("PegelAlarm water history is unavailable")
            if water.empty:
                raise RuntimeError("PegelAlarm water history is empty")

            latitude, longitude = resolve_station_coordinates(catalog, station_id)
            water_times = pd.to_datetime(water["sourceDate"], utc=True)
            weather = fetch_inca(
                station_id,
                latitude,
                longitude,
                water_times.min().to_pydatetime(),
                water_times.max().to_pydatetime(),
            )
            weather.to_parquet(weather_path, index=False)
            print(f"Saved {len(weather):,} weather observations to {weather_path}")
        latest_weather = weather
    except Exception as error:
        failures[str(weather_path)] = error
        print(f"Failed {weather_path}: {error}")

if latest_weather is not None:
    latest_weather.info()
    display(latest_weather.head())
else:
    print("No weather history DataFrame is available to display.")

In [ ]:
if failures:
    raise RuntimeError(summarize_failures(failures))

print("Raw-data fetch completed successfully.")